# 02 · Benchmark evaluation (Revisited Oxford & Paris)

Builds the embedding database for ROxford5k / RParis6k and evaluates it with the official revisitop protocol (mAP and mP@k for the Easy / Medium / Hard splits). This is the baseline configuration reported in Table 5.3 of the thesis.

Download the datasets first with `python -c "from benchmark.revisitop.download import download_datasets; download_datasets('.')"`.

In [ ]:
# Run from the repository root so that relative paths and package imports resolve
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [1]:
from landmark_detection.pipeline import Pipeline_Landmark_Detection
from landmark_detection.utils import show_image, show_bboxes, show_similarity_search
from benchmark.revisitop.evaluate import compute_map
from benchmark.revisitop.dataset import configdataset
from benchmark.evaluation import run_evaluation, run_evaluation2, show_inference_example

import os
import pandas as pd
import pickle
import numpy as np
import torch

In [2]:
VERSION = 0

# DATABASE
force_rebuild = False
save_every = 500
min_area = 0.0
min_sim_db = 0.0

# PIPELINE
detector_file = "yolov8n-oiv7.pt"
extractor_onnx_file = "cvnet-sg-v" + str(VERSION) + ".onnx"
pipeline_onnx_file = "pipeline-yolo-cvnet-sg-v" + str(VERSION) + ".onnx"
image_dim = (640, 640)
allowed_classes = [41,68,70,74,87,95,113,144,150,158,164,165,193,205,212,224,257,
                              298,310,335,351,354,390,393,401,403,439,442,457,466,489,510,512,
                              514,524,530,531,543,546,554,565,573,580,588,591]
score_thresh = 0.05
iou_thresh = 0.25
scales = [0.7071, 1.0, 1.4142]
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]
rgem_pr = 2.5
rgem_size = 5
gem_p = 4.6
sgem_ps = 10.0
sgem_infinity = False
eps = 1e-8
topk = 10
min_sim = 0.70
min_votes = 0.60
remove_inner_boxes = 0.5
join_boxes = True

In [3]:
pipeline = Pipeline_Landmark_Detection(
        detector_file = detector_file,
        extractor_onnx_file = extractor_onnx_file,
        pipeline_onnx_file = pipeline_onnx_file,
        image_dim = image_dim,
        allowed_classes = allowed_classes,
        score_thresh = score_thresh,
        iou_thresh = iou_thresh,
        scales = scales,
        mean = mean,
        std = std,
        rgem_pr = rgem_pr,
        rgem_size = rgem_size,
        gem_p = gem_p,
        sgem_ps = sgem_ps,
        sgem_infinity = sgem_infinity,
        eps = eps,
        topk = topk,
        min_sim = min_sim,
        min_votes = min_votes,
        remove_inner_boxes = remove_inner_boxes,
        join_boxes = join_boxes
)

Creando versión ONNX del preprocess
Instanciando el preprocessor
Creando versión ONNX del detector
Ultralytics 8.3.146  Python-3.11.11 torch-2.7.0+cpu CPU (Intel Core(TM) i5-9300HF 2.40GHz)
YOLOv8n summary (fused): 72 layers, 3,492,527 parameters, 0 gradients, 10.5 GFLOPs

PyTorch: starting from '<repo>\landmark_detection\models\yolov8n-oiv7.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 605, 8400) (6.9 MB)

ONNX: starting export with onnx 1.17.0 opset 16...
ONNX: slimming with onnxslim 0.1.54...
ONNX: export success  1.5s, saved as '<repo>\landmark_detection\models\yolov8n-oiv7.onnx' (13.5 MB)

Export complete (2.1s)
Results saved to <repo>\landmark_detection\models
Predict:         yolo predict task=detect model=<repo>\landmark_detection\models\yolov8n-oiv7.onnx imgsz=640  
Validate:        yolo val task=detect model=<repo>\landmark_detection\models\yolov8n-oiv7.onnx imgsz=640 data=/usr/src/ultralytics/ultralytics/cfg/datasets/open-images-v7.yaml  
Visualize:     

In [4]:
# Paths
CLASS_NAMES_PATH = os.path.join("configs", "oiv7.yaml")
PIPELINE_CONFIG_PATH = os.path.join("configs", "pipeline-yolo-cvnet-sg-v" + str(VERSION))
DATASETS_PATH = os.path.abspath("datasets")

# Dataset Oxford
ROXFORD5K_PATH = os.path.join(DATASETS_PATH, "roxford5k", "jpg")
ROXFORD5K_PKL = os.path.join(DATASETS_PATH, "roxford5k", "gnd_roxford5k.pkl")
ROXFORD5K_CSV = os.path.join(DATASETS_PATH, "roxford5k", "roxford5k_image_data.csv")
ROXFORD5K_DF = os.path.join(DATASETS_PATH, "roxford5k", "results", "df_roxford5k-v" + str(VERSION) + ".pkl")
ROXFORD5K_DESC = os.path.join(DATASETS_PATH, "roxford5k", "results", "desc_roxford5k-v" + str(VERSION) + ".pkl")

# Dataset Paris
RPARIS6K_PATH = os.path.join(DATASETS_PATH, "rparis6k", "jpg")
RPARIS6K_PKL = os.path.join(DATASETS_PATH, "rparis6k", "gnd_rparis6k.pkl")
RPARIS6K_CSV = os.path.join(DATASETS_PATH, "rparis6k", "rparis6k_image_data.csv")
RPARIS6K_DF = os.path.join(DATASETS_PATH, "rparis6k", "results", "df_rparis6k-v" + str(VERSION) + ".pkl")
RPARIS6K_DESC = os.path.join(DATASETS_PATH, "rparis6k", "results", "desc_rparis6k-v" + str(VERSION) + ".pkl")

In [5]:
pipeline.to_json(PIPELINE_CONFIG_PATH)

In [8]:
cfg = configdataset('roxford5k', DATASETS_PATH)

In [9]:
# Build/load roxford5k dataset
df_result_ox, descriptors_final_ox, places_db_ox = pipeline.build_image_database(
    image_folder = ROXFORD5K_PATH,
    df_pickle_path = ROXFORD5K_DF,
    descriptor_pickle_path = ROXFORD5K_DESC,
    return_places_db = True,
    force_rebuild = force_rebuild,
    save_every = save_every,
    min_area = min_area,
    min_sim = min_sim_db
)

Procesando imágenes: 0it [00:00, ?it/s]


In [10]:
# Evaluate roxford5k
roxford5k_eval = run_evaluation2(df_result_ox, places_db_ox, 'roxford5k')

>> roxford5k: Evaluating test dataset...
>> roxford5k: Loading features...
>> roxford5k: Retrieval...
>> roxford5k: mAP E: 75.49, M: 58.69, H: 30.34
>> roxford5k: mP@k[ 1  5 10] E: [      97.06       84.04       78.46], M: [      95.71       81.81       77.19], H: [      64.29       53.43       44.14]


In [11]:
# Evaluate roxford5k
roxford5k_eval = run_evaluation2(df_result_ox, places_db_ox, 'roxford5k', use_bbox=True)

>> roxford5k: Evaluating test dataset...
>> roxford5k: Loading features...
>> roxford5k: Retrieval...
>> roxford5k: mAP E: 73.42, M: 58.53, H: 33.16
>> roxford5k: mP@k[ 1  5 10] E: [      95.59       82.87       78.94], M: [      94.29       80.29       76.95], H: [      74.29       55.79       46.21]


In [ ]:
show_inference_example(
    df_result_ox,
    descriptors_final_ox,
    q_idx = [1,5,10,15,20,25,60,35,40,45],
    top_n = 9,
    dataset = "roxford5k",
    use_bbox = False,
)

In [ ]:
show_inference_example(
    df_result_ox,
    descriptors_final_ox,
    q_idx = [1,5,10,15,20,25,60,35,40,45],
    top_n = 9,
    dataset = "roxford5k",
    use_bbox = True,
)

In [12]:
# Build/load rparis6k dataset
df_result_pa, descriptors_final_pa, places_db_pa = pipeline.build_image_database(
    image_folder = RPARIS6K_PATH,
    df_pickle_path = RPARIS6K_DF,
    descriptor_pickle_path = RPARIS6K_DESC,
    return_places_db = True,
    force_rebuild = force_rebuild,
    save_every = save_every,
    min_area = min_area,
    min_sim = min_sim_db
)

Procesando imágenes: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6392/6392 [1:40:32<00:00,  1.06it/s]


In [13]:
# Evaluate rparis6k
rparis6k_eval = run_evaluation2(df_result_pa, places_db_pa, 'rparis6k')

>> rparis6k: Evaluating test dataset...
>> rparis6k: Loading features...
>> rparis6k: Retrieval...
>> rparis6k: mAP E: 88.19, M: 76.67, H: 55.09
>> rparis6k: mP@k[ 1  5 10] E: [      97.14       94.95       93.38], M: [      98.57       97.71       97.29], H: [      92.86          92       89.43]


In [ ]:
# Evaluate rparis6k
rparis6k_eval = run_evaluation2(df_result_pa, places_db_pa, 'rparis6k', use_bbox=True)

In [ ]:
show_inference_example(
    df_result_pa,
    descriptors_final_pa,
    q_idx = [1,5,10,15,20,25,60,35,40,45],
    top_n = 9,
    dataset = "rparis6k",
    use_bbox = False,
)